# Function Calling Utilities API Reference

Developer-facing statements defined in `libs/core/langchain_core/utils/function_calling.py`.

# `PYTHON_TO_JSON_TYPES`

Maps common Python type names to JSON Schema type names.

```python
PYTHON_TO_JSON_TYPES = {
    "str": "string",
    "int": "integer",
    "float": "number",
    "bool": "boolean",
}
```

---

# `FunctionDescription: TypedDict`

Represents a callable function that can be sent to an LLM.

```python
name: str # Function name
description: str # Function description
parameters: dict[str, Any] # Function parameter schema
```

---

# `ToolDescription: TypedDict`

Represents a function tool for the OpenAI API.

```python
type: Literal["function"] # Tool type
function: FunctionDescription # Function description
```

---

# `convert_to_openai_function`

Converts a supported function-like object into an OpenAI function-calling schema.

```python
convert_to_openai_function(
    function: Mapping[str, Any] | type | Callable[..., Any] | BaseTool, # Function dictionary, schema class, tool, or callable to convert
    *,
    strict: bool | None = None, # Whether to enforce exact JSON Schema matching
) -> dict[str, Any] # OpenAI-compatible function schema
```

Supported inputs include OpenAI function dictionaries, JSON Schemas with a top-level `title`, Anthropic tool dictionaries, Amazon Bedrock Converse tool dictionaries, Pydantic model classes, `TypedDict` classes, LangChain tools, and Python callables.

For existing OpenAI function dictionaries, only `name`, `description`, `parameters`, and `strict` are retained. JSON Schema inputs use `title` as the function name.

When `strict` is not `None`, it is added to the output. If `strict=True`, declared properties are made required and nested parameter schemas are updated with `additionalProperties=False`.

Raises `ValueError` when the input format is unsupported or when an existing `strict` value conflicts with the explicit argument.

Behavior changed in `langchain-core` 0.3.16: `description` and `parameters` became optional; only `name` is guaranteed.

---

# `convert_to_openai_tool`

Converts a supported tool-like object into an OpenAI tool schema.

```python
convert_to_openai_tool(
    tool: Mapping[str, Any] | type[BaseModel] | Callable[..., Any] | BaseTool, # Tool dictionary, schema class, LangChain tool, or callable to convert
    *,
    strict: bool | None = None, # Whether to enforce exact JSON Schema matching
) -> dict[str, Any] # OpenAI-compatible tool schema
```

Known OpenAI built-in tool dictionaries and `web_search_preview` variants are returned unchanged.

A LangChain `Tool` whose metadata contains `{"type": "custom_tool"}` is converted to a custom tool containing its name and description, plus `format` when present in the metadata.

Other inputs are converted with `convert_to_openai_function` and wrapped as:

```python
{"type": "function", "function": openai_function}
```

Behavior changed in `langchain-core` 0.3.16: function `description` and `parameters` became optional.

Behavior changed in `langchain-core` 0.3.44: OpenAI Responses API-style tool dictionaries are returned unchanged.

Behavior changed in `langchain-core` 0.3.63: OpenAI image-generation built-in tools are supported.

---

# `convert_to_json_schema`

Converts a supported schema representation into JSON Schema.

```python
convert_to_json_schema(
    schema: dict[str, Any] | type[BaseModel] | Callable[..., Any] | BaseTool, # Schema representation to convert
    *,
    strict: bool | None = None, # Whether to enforce exact JSON Schema matching during conversion
) -> dict[str, Any] # JSON Schema representation
```

The input is first converted with `convert_to_openai_tool`. The function name becomes the JSON Schema `title`; the function description is copied when present, and the parameter schema is merged into the result.

Raises `ValueError` when the converted input is not an OpenAI function tool containing a function name.

---

# `tool_example_to_messages`

Converts a structured-output example into chat messages suitable for an LLM.

```python
@beta()
tool_example_to_messages(
    input: str, # User input from which information is extracted
    tool_calls: list[BaseModel], # Pydantic model instances representing tool calls
    tool_outputs: list[str] | None = None, # Tool-result messages corresponding to the calls
    *,
    ai_response: str | None = None, # Optional final AI response
) -> list[BaseMessage] # Human, AI tool-call, tool-result, and optional final AI messages
```

The result begins with a `HumanMessage`, followed by an `AIMessage` containing one OpenAI-format function call per Pydantic model. Each call receives a generated UUID, uses the model class name as its function name, and serializes arguments with `model_dump_json()`.

When `tool_outputs` is absent or empty, each call receives the placeholder `"You have correctly called this tool."`. A `ToolMessage` is created for each paired output and tool call. A final `AIMessage` is appended when `ai_response` is truthy.

In [ ]:
from pydantic import BaseModel, Field # Import Pydantic model utilities

from langchain_core.utils.function_calling import convert_to_json_schema # Import JSON Schema converter
from langchain_core.utils.function_calling import convert_to_openai_function # Import OpenAI function converter
from langchain_core.utils.function_calling import convert_to_openai_tool # Import OpenAI tool converter
from langchain_core.utils.function_calling import tool_example_to_messages # Import tool-example message converter


class WeatherRequest(BaseModel): # Define structured arguments for a weather tool
    city: str = Field(description="City whose weather should be checked") # Store the city name
    unit: str = Field(default="celsius", description="Temperature unit") # Store the preferred temperature unit

In [ ]:
# Convert the model into an OpenAI function
openai_function = convert_to_openai_function( # Convert the Pydantic model into a function schema
    WeatherRequest, # Provide the model class to convert
    strict=True, # Require arguments to match the schema exactly
) # Finish the conversion

openai_function # Display the generated function schema in Jupyter

In [ ]:
# Convert the model into an OpenAI tool
openai_tool = convert_to_openai_tool( # Convert the model into an OpenAI tool definition
    WeatherRequest, # Provide the model class to convert
    strict=True, # Enable strict schema matching
) # Finish the tool conversion

openai_tool # Display the generated tool definition

In [ ]:
# Convert the model into standard JSON Schema
json_schema = convert_to_json_schema( # Convert the model into JSON Schema
    WeatherRequest, # Provide the model class
    strict=True, # Apply strict schema rules
) # Finish the conversion

json_schema # Display the generated JSON Schema

In [ ]:
# Create example tool-calling messages
weather_request = WeatherRequest( # Create an example structured tool call
    city="Delhi", # Specify the city
    unit="celsius", # Specify the temperature unit
) # Finish creating the tool-call arguments

messages = tool_example_to_messages( # Convert the example into LangChain messages
    input="What is the weather in Delhi?", # Provide the original user request
    tool_calls=[weather_request], # Provide the structured tool call
    tool_outputs=["The temperature in Delhi is 31°C."], # Provide the simulated tool result
    ai_response="Delhi is currently around 31°C.", # Provide an optional final AI response
) # Finish generating the messages

for message in messages: # Iterate through the generated message sequence
    print(type(message).__name__) # Display the message class
    print(message.content) # Display the message content
    print(message.additional_kwargs) # Display tool-call metadata when present
    print("-" * 40) # Separate each displayed message
